In [1]:
import pandas as pd
import sys
from pathlib import Path

In [2]:
customer_path = Path(
    r"C:\Users\YapJack\Desktop\Data Science\University\Personal Projects\data-warehouse-project\Files\bronze\customers"
)

geolocation_path = Path(
    r"C:\Users\YapJack\Desktop\Data Science\University\Personal Projects\data-warehouse-project\Files\bronze\geolocation"
)

sellers_path = Path(
    r"C:\Users\YapJack\Desktop\Data Science\University\Personal Projects\data-warehouse-project\Files\bronze\sellers"
)

products_path = Path(
    r"C:\Users\YapJack\Desktop\Data Science\University\Personal Projects\data-warehouse-project\Files\bronze\products"
)

order_payments_path = Path(
    r"C:\Users\YapJack\Desktop\Data Science\University\Personal Projects\data-warehouse-project\Files\bronze\order_payments"
)

order_reviews_path = Path(
    r"C:\Users\YapJack\Desktop\Data Science\University\Personal Projects\data-warehouse-project\Files\bronze\order_reviews"
)

orders_path = Path(
    r"C:\Users\YapJack\Desktop\Data Science\University\Personal Projects\data-warehouse-project\Files\bronze\orders"
)

order_items_path = Path(
    r"C:\Users\YapJack\Desktop\Data Science\University\Personal Projects\data-warehouse-project\Files\bronze\order_items"
)

product_category_translation_path = Path(
    r"C:\Users\YapJack\Desktop\Data Science\University\Personal Projects\data-warehouse-project\Files\bronze\product_category_translation"
)

customer_df = pd.read_parquet(customer_path)
geolocation_df = pd.read_parquet(geolocation_path)
sellers_df = pd.read_parquet(sellers_path)
products_df = pd.read_parquet(products_path)
order_payments_df = pd.read_parquet(order_payments_path)
order_reviews_df = pd.read_parquet(order_reviews_path)
orders_df = pd.read_parquet(orders_path)
order_items_df = pd.read_parquet(order_items_path)
product_category_translation_df = pd.read_parquet(
    product_category_translation_path
)

## Customer

No NULLs

In [3]:
customer_df.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [5]:
customer_df.duplicated().sum()

np.int64(0)

In [48]:
customer_df.dtypes

customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

In [49]:
customer_df.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


## Geolocation

The `geolocation` dataset contains multiple records with the same `geolocation_zip_code_prefix`.


### Cause

This is an expected characteristic of the Olist dataset and **not a data quality issue**.

The dataset stores only the first five digits of the Brazilian postal code (CEP), known as the **ZIP code prefix**. A single ZIP code prefix represents multiple addresses, each with slightly different latitude and longitude coordinates. As a result, multiple geolocation records can legitimately share the same `geolocation_zip_code_prefix`.

### Impact

The `customers` and `sellers` datasets reference locations using `geolocation_zip_code_prefix`. Since the geolocation table contains multiple records for the same prefix, it cannot be directly used as a lookup table without introducing one-to-many joins.

### Resolution

- **Bronze Layer**
  - Preserve the raw geolocation records without modification.

- **Silver Layer**
  - Create a deduplicated geolocation lookup table by grouping on `geolocation_zip_code_prefix`.
  - Calculate the average latitude and longitude for each ZIP code prefix.
  - Retain the first `geolocation_city` and `geolocation_state`, as they are expected to be consistent within each ZIP code prefix.

### Status

**Expected dataset characteristic** — No cleansing required in the Bronze layer. A deduplicated lookup table will be generated in the Silver layer to support consistent joins with customer and seller data.

In [5]:
geolocation_df["geolocation_zip_code_prefix"].is_unique

False

In [4]:
def deduplicate_geolocation(df: pd.DataFrame) -> pd.DataFrame:
    """
    Deduplicate geolocation records by ZIP code.

    Multiple coordinates for the same ZIP code are averaged.
    The first city and state are retained.
    """

    return (
        df.groupby("geolocation_zip_code_prefix", as_index=False)
        .agg(
            geolocation_lat=("geolocation_lat", "mean"),
            geolocation_lng=("geolocation_lng", "mean"),
            geolocation_city=("geolocation_city", "first"),
            geolocation_state=("geolocation_state", "first"),
        )
    )
duplciated_goelocation_df = deduplicate_geolocation(geolocation_df)

In [5]:
duplciated_goelocation_df["geolocation_zip_code_prefix"].is_unique

True

In [9]:
result = (
    customer_df.merge(
        geolocation_df,
        left_on="customer_zip_code_prefix",
        right_on="geolocation_zip_code_prefix",
        how="left",
        indicator=True
    )
)

missing_geolocation = result[result["_merge"] == "left_only"]

print(missing_geolocation)

                               customer_id                customer_unique_id  \
54126     ecb1725b26e8b8c458181455dfa434ea  b55a113bb84fc10eaf58c6d09ec69794   
60453     bcf86029aeed4ed8bac0e16eb14c22f5  7cd7974c9f79f75b77f323878ef87f43   
138245    f4302056f0c58570522590f8181de2c7  67b05b597a66b5c449025000b9430abb   
189878    03bbe0ce5c28e05f22917607db798818  8f3dca4306d5a89e4ae2c65c110603a2   
198010    ad4950aded55c2ea376be59506456d68  aa2b96dd03307ea6dc4b763c0b5f0b39   
...                                    ...                               ...   
14790976  cf818420383856a129134f5f8343f7b8  795c495a65f983b242fb01bd507977c5   
14837751  67f3e907dce402e696b15f9308ff22ed  6f232f2f5c7f33b7bd9d794d2afacadd   
14889266  f792e419335df11d82c32efcfb09c51b  c04c085b8e7573ba87b9ae1968d0985e   
15000610  78a11bb1fa72f556996b9a5b9bcd0629  e7536f62a200b415edd9491ac12a17fa   
15058843  ff09fd7b29e7488a8d8a20badcd8befe  8c21dd8c37144807c601f99f2a209dfb   

          customer_zip_code_prefix     

In [50]:
geolocation_df.dtypes

geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

In [117]:
duplciated_goelocation = geolocation_df[geolocation_df.duplicated(subset = ["geolocation_zip_code_prefix", "geolocation_city"])]
duplciated_goelocation.sort_values(by="geolocation_zip_code_prefix")

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
206,1001,-23.550498,-46.634338,sao paulo,SP
1435,1001,-23.549292,-46.633559,sao paulo,SP
299,1001,-23.549698,-46.633909,sao paulo,SP
235,1001,-23.550642,-46.634410,sao paulo,SP
326,1001,-23.551427,-46.634074,sao paulo,SP
...,...,...,...,...,...
999796,99980,-28.388342,-51.845194,david canabarro,RS
999808,99980,-28.387842,-51.846890,david canabarro,RS
999764,99980,-28.386239,-51.847741,david canabarro,RS
1000161,99980,-28.388932,-51.846871,david canabarro,RS


In [131]:
geolocation_df["geolocation_zip_code_prefix"].isna().sum()

np.int64(0)

In [12]:
state_city_count = (
    geolocation_df
    .groupby("geolocation_state")["geolocation_city"]
    .nunique()
)

print(state_city_count)


geolocation_state
AC      34
AL     130
AM      74
AP      17
BA     652
CE     260
DF      28
ES     160
GO     384
MA     299
MG    1426
MS     133
MT     213
PA     219
PB     254
PE     267
PI     278
PR     651
RJ     245
RN     214
RO      83
RR      14
RS     691
SC     420
SE      96
SP    1048
TO     173
Name: geolocation_city, dtype: int64


In [6]:
state_cities = (
    geolocation_df
    .groupby("geolocation_state")["geolocation_city"]
    .unique()
    .reset_index()
)

display(state_cities)

,geolocation_state,geolocation_city
0,AC,"[sao paulo, rio de janeiro, sena madureira, ri..."
1,AL,"[maceio, maceió, maceia³, barra de sao miguel,..."
2,AM,"[manaus, parintins, itacoatiara, silves, rio p..."
3,AP,"[macapa, serra do navio, laranjal do jari, mac..."
4,BA,"[salvador, salvador , lauro de freitas, madre ..."
5,CE,"[fortaleza, caucaia, eusebio, eusébio, aquiraz..."
6,DF,"[brasilia, brasília, cruzeiro, guara, guará, p..."
7,ES,"[vitória, vitoria, vila velha, serra, cariacic..."
8,GO,"[novo gama, luziania, valparaiso de goias, cid..."
9,MA,"[sao luis, são luís, santa rita, paco do lumia..."


In [5]:

VALID_BRAZIL_STATES = {
    "AC", "AL", "AP", "AM", "BA", "CE", "DF",
    "ES", "GO", "MA", "MT", "MS", "MG", "PA",
    "PB", "PR", "PE", "PI", "RJ", "RN", "RS",
    "RO", "RR", "SC", "SP", "SE", "TO"
}

def validate_state_codes(df: pd.DataFrame):

    # if "geolocation_state" not in df.columns:
    #     return {"passed": True, "invalid_rows": []}

    invalid = df[
        ~df["geolocation_state"].isin(VALID_BRAZIL_STATES)
    ]

    return {
        "passed": invalid.empty,
        "invalid_rows": invalid.index.tolist()
    }

results = validate_state_codes(geolocation_df)
print(f"{results["passed"]}: {results["invalid_rows"]}")

True: []


In [7]:
merged = pd.merge(
    customer_df,
    geolocation_df,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="inner"
)

customer_geo = (
    merged
    .groupby("customer_id")["geolocation_zip_code_prefix"]
    .nunique()
)

print(customer_geo[customer_geo > 1])

Series([], Name: geolocation_zip_code_prefix, dtype: int64)


In [9]:
missing_geo = (
    sellers_df.merge(
        geolocation_df,
        left_on="seller_zip_code_prefix",
        right_on="geolocation_zip_code_prefix",
        how="left"
    )
)

missing_geo = missing_geo[
    missing_geo["geolocation_zip_code_prefix"].isna()
]

missing_geo

,seller_id,seller_zip_code_prefix,seller_city,seller_state,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
63658,5962468f885ea01a1b6a97a218797b0a,82040,curitiba,PR,NaN,NaN,NaN,NaN,NaN
112299,2aafae69bf4c41fbd94053d9413e87ee,91901,porto alegre,RS,NaN,NaN,NaN,NaN,NaN
241204,2a50b7ee5aebecc6fd0ff9784a4747d6,72580,brasilia,DF,NaN,NaN,NaN,NaN,NaN
276433,2e90cb1677d35cfe24eef47d441b7c87,2285,sao paulo,SP,NaN,NaN,NaN,NaN,NaN
309371,0b3f27369a4d8df98f7eb91077e438ac,7412,aruja,SP,NaN,NaN,NaN,NaN,NaN
419156,42bde9fef835393bb8a8849cb6b7f245,71551,brasilia,DF,NaN,NaN,NaN,NaN,NaN
425736,870d0118f7a9d85960f29ad89d5d989a,37708,pocos de caldas,MG,NaN,NaN,NaN,NaN,NaN


## sellers

In [104]:
len(sellers_df)

3095

### Sellers null value analysis

In [102]:
sellers_df.isna().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [51]:
sellers_df.dtypes

seller_id                 object
seller_zip_code_prefix     int64
seller_city               object
seller_state              object
dtype: object

In [52]:
sellers_df

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
...,...,...,...,...
3090,98dddbc4601dd4443ca174359b237166,87111,sarandi,PR
3091,f8201cab383e484733266d1906e2fdfa,88137,palhoca,SC
3092,74871d19219c7d518d0090283e03c137,4650,sao paulo,SP
3093,e603cf3fec55f8697c9059638d6c8eb5,96080,pelotas,RS


## Products

In [32]:
products_df

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0


### Product null value analysis

In [105]:
products_df.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [17]:
products_df[products_df["product_category_name"].notna()]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0


In [27]:
result = (
    order_items_df
    .merge(
        products_df,
        on="product_id",
        how="left"
    )
)
result[result["product_category_name"].isna()]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
123,0046e1d57f4c07c8c92ab26be8c3dfc0,1,ff6caf9340512b8bf6d2a2a6df032cfa,38e6dada03429a47197d5d584d793b41,2017-10-02 15:49:17,7.79,7.78,None,NaN,NaN,NaN,200.0,16.0,5.0,12.0
125,00482f2670787292280e0a8153d82467,1,a9c404971d1a5b1cbc2e4070e02731fd,702835e4b785b67a084280efca355756,2017-02-17 16:18:07,7.60,10.96,None,NaN,NaN,NaN,700.0,35.0,14.0,11.0
132,004f5d8f238e8908e6864b874eda3391,1,5a848e4ab52fd5445cdc07aab1c40e48,c826c40d7b19f62a09e2d7c5e7295ee2,2018-03-06 09:29:25,122.99,15.61,None,NaN,NaN,NaN,400.0,20.0,12.0,15.0
142,0057199db02d1a5ef41bacbf41f8f63b,1,41eee23c25f7a574dfaf8d5c151dbb12,e5a3438891c0bfdb9394643f95273d8e,2018-01-25 09:07:51,20.30,16.79,None,NaN,NaN,NaN,200.0,16.0,2.0,11.0
171,006cb7cafc99b29548d4f412c7f9f493,1,e10758160da97891c2fdcbc35f0f031d,323ce52b5b81df2cd804b017b7f09aa7,2018-02-22 13:35:28,56.00,14.14,None,NaN,NaN,NaN,2200.0,16.0,2.0,11.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112306,ff24fec69b7f3d30f9dc1ab3aee7c179,1,5a848e4ab52fd5445cdc07aab1c40e48,c826c40d7b19f62a09e2d7c5e7295ee2,2018-02-01 02:40:12,122.99,15.61,None,NaN,NaN,NaN,400.0,20.0,12.0,15.0
112333,ff3024474be86400847879103757d1fd,1,f9b1795281ce51b1cf39ef6d101ae8ab,3771c85bac139d2344864ede5d9341e3,2017-11-21 03:55:39,39.90,9.94,None,NaN,NaN,NaN,400.0,32.0,15.0,15.0
112350,ff3a45ee744a7c1f8096d2e72c1a23e4,1,b61d1388a17e3f547d2bc218df02335b,07017df32dc5f2f1d2801e579548d620,2017-05-10 10:15:19,139.00,21.42,None,NaN,NaN,NaN,350.0,16.0,6.0,11.0
112438,ff7b636282b98e0aa524264b295ed928,1,431df35e52c10451171d8037482eeb43,6cd68b3ed6d59aaa9fece558ad360c0a,2018-02-22 15:35:35,49.90,15.11,None,NaN,NaN,NaN,475.0,21.0,15.0,21.0


In [28]:
result["product_category_name"].isna().sum()

np.int64(1603)

## Order Payments

order payments: composite key(order_id, payment_sequential)

In [29]:
duplciated_order_payments = order_payments_df[order_payments_df["order_id"].duplicated(keep=False)].sort_values("order_id")

duplciated_order_payments.head(30)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
80856,0016dfedd97fc2950e388d2971d718c7,2,voucher,1,17.92
89575,0016dfedd97fc2950e388d2971d718c7,1,credit_card,5,52.63
20036,002f19a65a2ddd70a090297872e6d64e,1,voucher,1,44.11
98894,002f19a65a2ddd70a090297872e6d64e,2,voucher,1,33.18
30155,0071ee2429bc1efdc43aa3e073a5290e,2,voucher,1,92.44
10244,0071ee2429bc1efdc43aa3e073a5290e,1,voucher,1,100.00
16459,009ac365164f8e06f59d18a08045f6c4,2,voucher,1,4.50
15298,009ac365164f8e06f59d18a08045f6c4,6,voucher,1,4.17
32058,009ac365164f8e06f59d18a08045f6c4,4,voucher,1,5.45
285,009ac365164f8e06f59d18a08045f6c4,5,voucher,1,8.75


In [7]:
order_payments_df.isna().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

## Order Reviews

The order_reviews dataset does not follow a one-to-one relationship between reviews and orders.

- A single review_id can be linked to multiple order_ids. 
- Likewise, a single order_id can have multiple review_ids. (updated maybe or second review)
- Analysis of records with the same review_id shows that the associated orders belong to the same customer (customer_unique_id) and share the same review score, review message, and review creation date. This suggests that the source system allows a single customer review to be associated with multiple orders, rather than these records being duplicate entries.

Although neither review_id nor order_id is unique on its own, validation confirms that the combination of (review_id, order_id) is unique across the dataset.

Conclusion: The composite key (review_id, order_id) is used as the natural primary key for the order_reviews table, as it uniquely identifies each review-order relationship while preserving the integrity of the original source data.

In [75]:
# remove duplicate order_ids and keep only one review per order, 
# keep the latest review based on review_creation_date.
order_reviews_df = (
    order_reviews_df
    .sort_values("review_creation_date")
    .drop_duplicates(subset="order_id", keep="last")
)

In [77]:
# Check full duplicated rows
order_reviews_df[order_reviews_df["review_id"].duplicated(keep=False)].sort_values(by="review_id")

# order_reviews_df[order_reviews_df["review_id"].duplicated(keep=False)]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,None,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,None,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,None,None,2017-09-21 00:00:00,2017-09-26 03:27:47
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,None,None,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,None,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
...,...,...,...,...,...,...,...
92062,fd582f520c76d0b29106fcef19d868fc,78668b283d34d8c72670a44109fbbe09,4,None,None,2017-10-26 00:00:00,2017-10-30 14:55:42
31120,fe5c833752953fed3209646f1f63b53c,4863e15fa53273cc7219c58f5ffda4fb,1,None,"Comprei dois produtos e ambos, mesmo enviados ...",2018-02-28 00:00:00,2018-02-28 13:57:52
40378,fe5c833752953fed3209646f1f63b53c,d3775e436e60258e62e678a0f68a0f8d,1,None,"Comprei dois produtos e ambos, mesmo enviados ...",2018-02-28 00:00:00,2018-02-28 13:57:52
82521,ff2fc9e68f8aabfbe18d710b83aabd30,1078d496cc6ab9a8e6f2be77abf5091b,2,None,None,2018-03-17 00:00:00,2018-03-19 11:44:15


In [54]:
order_reviews_df[order_reviews_df["review_comment_title"].notna()]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
9,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelh...,2018-05-22 00:00:00,2018-05-23 16:45:47
15,3948b09f7c818e2d86c9a546758b2335,e51478e7e277a83743b6f9991dbfa3fb,5,Super recomendo,"Vendedor confiável, produto ok e entrega antes...",2018-05-23 00:00:00,2018-05-24 03:00:01
19,373cbeecea8286a2b66c97b1b157ec46,583174fbe37d3d5f0d6661be3aad1786,1,Não chegou meu produto,Péssimo,2018-08-15 00:00:00,2018-08-15 04:10:37
22,d21bbc789670eab777d27372ab9094cc,4fc44d78867142c627497b60a7e0228a,5,Ótimo,Loja nota 10,2018-07-10 00:00:00,2018-07-11 14:10:25
34,c92cdd7dd544a01aa35137f901669cdf,37e7875cdce5a9e5b3a692971f370151,4,Muito bom.,Recebi exatamente o que esperava. As demais en...,2018-06-07 00:00:00,2018-06-09 18:44:02
...,...,...,...,...,...,...,...
99192,0e7bc73fde6782891898ea71443f9904,bd78f91afbb1ecbc6124974c5e813043,4,👍,Aprovado!,2018-07-04 00:00:00,2018-07-05 00:25:13
99196,58be140ccdc12e8908ff7fd2ba5c7cb0,0ebf8e35b9807ee2d717922d5663ccdb,5,muito bom produto,"Ficamos muito satisfeitos com o produto, atend...",2018-06-30 00:00:00,2018-07-02 23:09:35
99197,51de4e06a6b701cb2be47ea0e689437b,b7467ae483dbe956fe9acdf0b1e6e3f4,3,Não foi entregue o pedido,Bom dia \r\nDas 6 unidades compradas só recebi...,2018-06-05 00:00:00,2018-06-06 10:52:19
99199,40743b46a0ee86375cedb95e82b78d75,3e93213bb8fdda91186b4018b2fe0030,5,OTIMA EMBALAGEM,None,2018-08-08 00:00:00,2018-08-08 16:56:16


In [87]:
result = (
    order_reviews_df
    .merge(
        orders_df[["order_id", "customer_id"]],
        on="order_id",
        how="left"
    )
    .merge(
        customer_df[["customer_id", "customer_unique_id"]],
        on="customer_id",
        how="left"
    )
    [[
        "review_id",
        "order_id",
        "customer_unique_id",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date",
    ]]
)

result[result["review_id"].duplicated(keep=False)].sort_values(by="review_id").head(30)

,review_id,order_id,customer_unique_id,review_score,review_comment_title,review_comment_message,review_creation_date
56180,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,f30856ad31d3e74253a3f4ccef670648,1,None,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00
56271,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,f30856ad31d3e74253a3f4ccef670648,1,None,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00
24279,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,91ffaf5eb7f4bd6a48b07f3546fb7a99,5,None,None,2017-09-21 00:00:00
24236,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,91ffaf5eb7f4bd6a48b07f3546fb7a99,5,None,None,2017-09-21 00:00:00
56181,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,ba16ab3721ffd6b57fc87a7a4f6a4fcf,1,None,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00
56209,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,ba16ab3721ffd6b57fc87a7a4f6a4fcf,1,None,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00
55313,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,d5ca668597a3b40c1e527db8c761f03c,5,None,None,2018-03-02 00:00:00
55407,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,d5ca668597a3b40c1e527db8c761f03c,5,None,None,2018-03-02 00:00:00
22478,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,4c45ebf94ecf4107850d91bc326851c3,1,None,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00
22475,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,4c45ebf94ecf4107850d91bc326851c3,1,None,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00


In [92]:
order_reviews_df.duplicated(
    subset=["review_id", "order_id"]
).any()


np.False_

In [8]:
order_reviews_df.isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [10]:
order_reviews_df[order_reviews_df["review_comment_message"].isna()]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,None,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,None,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,None,None,2018-02-17 00:00:00,2018-02-18 14:36:24
5,15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,None,None,2018-04-13 00:00:00,2018-04-16 00:39:37
6,07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,None,None,2017-07-16 00:00:00,2017-07-18 19:30:34
...,...,...,...,...,...,...,...
99217,c6b270c61f67c9f7cb07d84ea8aeaf8b,48f7ee67313eda32bfcf5b9c1dd9522d,5,None,None,2017-12-13 00:00:00,2017-12-14 11:09:36
99218,af2dc0519de6e0720ef0c74292fb4114,d699c734a0b1c8111f2272a3f36d398c,5,None,None,2018-04-27 00:00:00,2018-04-30 01:18:57
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,None,None,2018-07-07 00:00:00,2018-07-14 17:18:30
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,None,None,2017-12-09 00:00:00,2017-12-11 20:06:42


## Order Items

order_items: composite key(order_id, order_item_id)

In [43]:
order_items_df.sort_values(by="order_item_id")

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
112628,fff3983dfa3c5a0d752d8d17baa406a0,1,092be1e8336fc404c57bd5970d056886,cbd996ad3c1b7dc71fd0e5f5df9087e2,2018-07-16 22:05:13,66.39,14.05
112629,fff60e5408a9dd1e92ee30023052af30,1,1a405418406359cc2b8815f93bf359c2,4d6d651bd7684af3fffabd5f08d12e5a,2018-02-05 10:55:55,129.90,18.80
112630,fff6889749958e42b47a7977a4cf0ea0,1,75f6a4f019ec1322758d53b2fee2cc12,058cb5aeb36d7c0fcae20fc85d5e0a59,2017-10-01 22:56:15,92.00,31.60
112631,fff6b8ca971f8e3ec822e99d0f2d3d21,1,9afaad66aca8b0c79e4f084a89c9c92b,42bde9fef835393bb8a8849cb6b7f245,2017-09-21 12:04:29,199.00,16.83
112632,fff7c4452f050315db1b3f24d9df5fcd,1,dd469c03ad67e201bc2179ef077dcd48,7e93a43ef30c4f03f38b393420bc753a,2017-06-07 17:05:23,736.00,20.91
...,...,...,...,...,...,...,...
57315,8272b63d03f5f79c56e9e4120aec44ef,19,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.20,7.89
11951,1b15974a0141d54e36626dca3fdc731a,20,ee3d532c8a438679776d222e997606b3,8e6d7754bc7e0f22c96d255ebda59eba,2018-03-01 02:50:48,100.00,10.12
75122,ab14fdcfbe524636d65ee38360e22ce8,20,9571759451b1d780ee7c15012ea109d4,ce27a3cc3c8cc1ea79d11e561e9bebb6,2017-08-30 14:30:23,98.70,14.44
57316,8272b63d03f5f79c56e9e4120aec44ef,20,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.20,7.89


In [36]:
order_items_df.count()

order_id               112650
order_item_id          112650
product_id             112650
seller_id              112650
shipping_limit_date    112650
price                  112650
freight_value          112650
dtype: int64

In [11]:
order_items_df.isna().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

## product_category_translation

In [12]:
product_category_translation_df.isna().sum()

product_category_name            0
product_category_name_english    0
dtype: int64

## Orders

order_id is pk

In [94]:
orders_df["order_id"].is_unique

True

In [58]:
order_customer_df = pd.merge(
    orders_df,
    customer_df,
    on="customer_id",
    how="inner"
)

frequent_cust = customer_df["customer_unique_id"].value_counts().index[0]

order_customer_df[order_customer_df["customer_unique_id"] == frequent_cust]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
2340,c2213109a2cc0e75d55585b7aaac6d97,897b7f72042714efaa64ac306ba0cafc,delivered,2018-08-07 23:32:14,2018-08-07 23:45:21,2018-08-09 13:35:00,2018-08-10 20:26:44,2018-08-13 00:00:00,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
3660,23427a6bd9f8fd1b51f1b1e5cc186ab8,a8fabc805e9a10a3c93ae5bff642b86b,delivered,2018-05-21 22:44:31,2018-05-22 01:53:35,2018-05-22 14:18:00,2018-05-23 15:33:09,2018-05-29 00:00:00,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
5167,e3071b7624445af6e4f3a1b23718667d,0bf8bf19944a7f8b40ba86fef778ca7c,delivered,2017-09-05 22:14:52,2017-09-05 22:30:56,2017-09-06 15:26:12,2017-09-11 13:27:49,2017-09-22 00:00:00,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
11694,d3582fd5ccccd9cb229a63dfb417c86f,a682769c4bc10fc6ef2101337a6c83c9,delivered,2018-08-20 19:14:26,2018-08-20 19:30:05,2018-08-21 15:11:00,2018-08-24 14:08:43,2018-09-04 00:00:00,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
16231,5837a2c844decae8a778657425f6d664,31dd055624c66f291578297a551a6cdf,unavailable,2017-07-17 22:11:13,2017-07-17 22:23:46,None,None,2017-08-17 00:00:00,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
19127,bf92c69b7cc70f7fc2c37de43e366173,42dbc1ad9d560637c9c4c1533746f86d,delivered,2017-07-24 22:11:50,2017-07-24 22:25:14,2017-07-26 01:42:03,2017-07-31 16:59:58,2017-08-15 00:00:00,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
24886,6bdf325f0966e3056651285c0aed5aad,6289b75219d757a56c0cce8d9e427900,delivered,2018-05-22 23:08:55,2018-05-22 23:36:01,2018-05-23 19:02:00,2018-05-24 11:58:23,2018-05-30 00:00:00,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
33703,4f62d593acae92cea3c5662c76122478,dfb941d6f7b02f57a44c3b7c3fefb44b,delivered,2017-07-18 23:10:58,2017-07-18 23:23:26,2017-07-20 19:00:02,2017-07-21 16:19:40,2017-07-31 00:00:00,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
39449,b850a16d8faf65a74c51287ef34379ce,1bd3585471932167ab72a84955ebefea,delivered,2017-11-22 20:01:53,2017-11-22 20:12:32,2017-11-24 16:07:56,2017-11-27 18:49:13,2017-12-04 00:00:00,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
58811,519203404f6116d406a970763ee75799,1c62b48fb34ee043310dcb233caabd2e,delivered,2017-08-05 08:59:43,2017-08-05 09:10:13,2017-08-07 18:50:00,2017-08-09 15:22:28,2017-08-25 00:00:00,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP


In [14]:
orders_df.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64